# AIStylist Buddy: Virtual Try-On server (IDM-VTON on Colab)

Run the cells top to bottom. The last cell prints a public link like `https://xxxx.gradio.live`.
Put that link in your app as `TRYON_URL` (in `.streamlit/secrets.toml`) and open the **Virtual Try-On** page.

Before you start: **Runtime > Change runtime type > T4 GPU** (or better).
The first run takes roughly 15 to 25 minutes because it installs packages and downloads the model weights (about 10 GB).
Keep this tab open while you use the app. If Colab disconnects, run the last cell again and update `TRYON_URL` with the new link.

In [2]:
# 1. Check the GPU
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
# 2. Get the IDM-VTON code
%cd /content
!git clone https://github.com/yisol/IDM-VTON.git
%cd /content/IDM-VTON

/content
Cloning into 'IDM-VTON'...
remote: Enumerating objects: 1502, done.
remote: Counting objects: 100% (264/264), done.
remote: Compressing objects: 100% (184/184), done.
remote: Total 1502 (delta 103), reused 80 (delta 80), pack-reused 1238 (from 2)
Receiving objects: 100% (1502/1502), 22.19 MiB | 14.08 MiB/s, done.
Resolving deltas: 100% (407/407), done.
/content/IDM-VTON


In [6]:
!pip install -q "huggingface_hub==0.25.2" diffusers==0.25.0 transformers==4.36.2 accelerate==0.25.0 \
    einops scipy opencv-python fvcore cloudpickle omegaconf pycocotools av onnxruntime-gpu \
    "numpy<2" "gradio>=4.36,<5" torchmetrics tqdm

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.6/436.6 kB 4.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 78.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 11.0 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 49.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.7/246.7 MB 6.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 71.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 25.3 MB/s eta 0:

In [ ]:
# 4. Download the pose / parsing / DensePose checkpoints that IDM-VTON needs
from huggingface_hub import hf_hub_download
files = [
    "ckpt/densepose/model_final_162be9.pkl",
    "ckpt/humanparsing/parsing_atr.onnx",
    "ckpt/humanparsing/parsing_lip.onnx",
    "ckpt/openpose/ckpts/body_pose_model.pth",
]
for f in files:
    p = hf_hub_download(repo_id="yisol/IDM-VTON", repo_type="space", filename=f, local_dir="/content/IDM-VTON")
    print("ok", p)

In [ ]:
# 5. Make the demo create a public link
import re, pathlib
p = pathlib.Path("/content/IDM-VTON/gradio_demo/app.py")
s = p.read_text()
s = re.sub(r"image_blocks\.launch\([^)]*\)", "image_blocks.queue(max_size=5).launch(share=True)", s)
p.write_text(s)
print("patched" if "share=True" in s else "COULD NOT PATCH: edit the last line of gradio_demo/app.py by hand")

In [ ]:
# 6. Start the server. Wait for the line that says: Running on public URL: https://....gradio.live
# Copy that link into TRYON_URL in your app. Leave this cell running.
%cd /content/IDM-VTON/gradio_demo
!python app.py